# Notebook 4: Measuring inference resistance

## Differential opacity

$$\Delta O = \text{Python accuracy} - \text{PoC accuracy}$$

1.0 means perfect on Python but 0% on PoC; 0.0 means no difference.

This notebook loads `experiments_v2/results/MASTER_RESULTS.json` and reproduces:

1. RQ1 blind-test accuracy per model (bar chart)
2. Differential opacity for models with both PoC and Python scores
3. Per-keyword confusion ranking
4. Mechanism ablation curve (baseline → full PoC)

In [ ]:
# Setup
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

RESULTS_PATH = os.path.join(
    os.path.abspath('..'), 'experiments_v2', 'results', 'MASTER_RESULTS.json'
)

with open(RESULTS_PATH) as fh:
    data = json.load(fh)

print('Keys in MASTER_RESULTS.json:', list(data.keys()))
meta = data.get('metadata', {})
print(f'\nGenerated : {meta.get("generated", "N/A")}')
print(f'Corpus    : {meta.get("corpus_size", "N/A")} programs')
print(f'Measurements: {meta.get("total_measurements", "N/A")}')

In [ ]:
# ── RQ1 Blind Test Table ──────────────────────────────────────────────────────

rq1     = data['rq1_blind_test']
models  = rq1['models']        # dict: model_name -> {accuracy, std, ci95, ...}
ranking = rq1['rankings']      # list, best first
mean_acc = rq1['mean_accuracy']

print(f'RQ1 — PoC Blind-Test Accuracy  (N=11 models, mean={mean_acc:.1%})')
print(f'{"-"*55}')
print(f'{"Rank":<5} {"Model":<30} {"Accuracy":>9} {"95% CI":>20}')
print(f'{"-"*55}')

for rank, name in enumerate(ranking, 1):
    m    = models[name]
    acc  = m['accuracy']
    ci   = m.get('ci95', [None, None])
    ci_s = f'[{ci[0]:.3f}, {ci[1]:.3f}]' if ci[0] is not None else 'N/A'
    print(f'{rank:<5} {name:<30} {acc:>8.1%}   {ci_s:>20}')

print(f'{"-"*55}')
print(f'{"":<5} {"Mean":<30} {mean_acc:>8.1%}')

In [ ]:
# ── Bar chart: PoC blind-test accuracy per model ─────────────────────────────

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    MATPLOTLIB = True
except ImportError:
    MATPLOTLIB = False
    print('matplotlib not installed — skipping plots.')
    print('Install with: pip install matplotlib')

if MATPLOTLIB:
    names  = ranking
    accs   = [models[n]['accuracy'] for n in names]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(range(len(names)), [a * 100 for a in accs], color='steelblue', edgecolor='white')

    # Mark mean
    ax.axhline(mean_acc * 100, color='crimson', linestyle='--', linewidth=1.5, label=f'Mean = {mean_acc:.1%}')

    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Exact-match accuracy (%)')
    ax.set_title('RQ1: PoC Blind-Test Accuracy per Model')
    ax.set_ylim(0, 100)
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig1_blind_test_accuracy.png', dpi=150)
    plt.show()
    print('Saved: fig1_blind_test_accuracy.png')

In [ ]:
# ── Differential Opacity: Python vs PoC ──────────────────────────────────────
# RQ2 only has Python scores for 3 models: GPT-4o-mini, GPT-4o, o4-mini.
# Map to their PoC scores from RQ1.

rq2_python = data['rq2_python_control']['models']

# rq1 model key mapping (rq2 uses slightly different names)
RQ2_TO_RQ1 = {
    'GPT-4o-mini': 'GPT-4o-mini',
    'GPT-4o':      'GPT-4o',
    'o4-mini':     'o4-mini',
}

print(f'Differential Opacity  (Python accuracy − PoC accuracy)')
print(f'{"-"*55}')
print(f'{"Model":<20} {"Python":>8} {"PoC":>8} {"Delta":>8}')
print(f'{"-"*55}')

deltas = {}
for rq2_name, rq1_name in RQ2_TO_RQ1.items():
    python_acc = rq2_python[rq2_name]['accuracy']
    poc_acc   = models[rq1_name]['accuracy']
    delta      = python_acc - poc_acc
    deltas[rq2_name] = delta
    print(f'{rq2_name:<20} {python_acc:>7.1%} {poc_acc:>7.1%} {delta:>+7.1%}')

print(f'{"-"*55}')
mean_delta = sum(deltas.values()) / len(deltas)
print(f'{"Mean":<20} {"":>7}  {"":>7}  {mean_delta:>+7.1%}')

In [ ]:
# ── Per-keyword confusion ranking ─────────────────────────────────────────────
# Lower accuracy = more confusion = more opacity.

per_kw = rq1['mean_per_keyword']   # dict: keyword -> mean accuracy across all models

# Sort ascending (lowest accuracy = most confusing keyword)
sorted_kw = sorted(per_kw.items(), key=lambda kv: kv[1])

print('Per-keyword confusion ranking  (ascending accuracy = more confusion)')
print(f'{"-"*45}')
print(f'{"Rank":<5} {"Keyword":<15} {"Mean Accuracy":>14}')
print(f'{"-"*45}')

for rank, (kw, acc) in enumerate(sorted_kw, 1):
    marker = '  <-- 100% confusion' if acc == 0.0 else ''
    print(f'{rank:<5} {kw:<15} {acc:>13.1%}{marker}')

zero_confusion = [kw for kw, acc in per_kw.items() if acc == 0.0]
print(f'\nKeywords at 0% accuracy (100% confusion): {zero_confusion}')

In [ ]:
# ── Mechanism ablation curve ───────────────────────────────────────────────────
# Shows how stacking mechanisms progressively reduces model accuracy.
# Data: mechanism_ablation from MASTER_RESULTS.json

ablation = data['mechanism_ablation']

# Ordered ablation stages
stages = ['python', 'baseline', '+sinks', '+mangling', '+semantic', '+all']
labels = ['Python', 'Baseline\n(no mech)', '+Gradient\nSinks', '+Tokenizer\nMangling', '+Semantic\nMisdirection', 'Full PoC\n(all)']

print('Mechanism ablation — accuracy at each stage')
print(f'{"-"*60}')
header = f'{"Stage":<22}' + ''.join(f'{m:<18}' for m in ablation)
print(header)
print(f'{"-"*60}')

for stage in stages:
    row = f'{stage:<22}'
    for model_name in ablation:
        val = ablation[model_name].get(stage, float('nan'))
        row += f'{val:<18.1%}'
    print(row)

if MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#e06c75', '#61afef']
    for idx, (model_name, stage_vals) in enumerate(ablation.items()):
        ys = [stage_vals.get(s, float('nan')) * 100 for s in stages]
        ax.plot(range(len(stages)), ys, marker='o', label=model_name, color=colors[idx], linewidth=2)

    ax.set_xticks(range(len(stages)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Exact-match accuracy (%)')
    ax.set_title('Mechanism Ablation: Stacking Opacity Mechanisms')
    ax.legend()
    ax.set_ylim(0, 110)
    plt.tight_layout()
    plt.savefig('fig2_ablation_curve.png', dpi=150)
    plt.show()
    print('Saved: fig2_ablation_curve.png')

## Summary

Full experiment (11 models, 40 corpus programs, 5,275 measurements):

- Mean PoC accuracy: 18% across all 11 models
- GPT-4o-mini: Python 75% → PoC 0% (75-point drop)
- `async`, `await`, `del` reached 100% confusion across all models tested
- Ablation: each mechanism adds opacity independently; full PoC collapses even the strongest
  models (o4-mini: 57% → near 0% at `+all`)
- Finetune resistance: after 100 training pairs, Qwen-Coder-1.5B recovers from 4% → 29%,
  below the 80% Python baseline